In [ ]:
!pip -q install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.4/94

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import mlflow

from pathlib import Path
from google.colab import files

In [ ]:
model_upload = files.upload()
model_filename = list(model_upload.keys()[0])

Saving house_price_model.pkl to house_price_model (1).pkl


In [ ]:
sample_upload = files.upload()
sample_filename = list(sample_upload.keys())[0]

Saving sample_house.csv to sample_house (1).csv


In [ ]:
sample_house =pd.read_csv(sample_filename)
sample_house.head()

,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished


In [ ]:
model = joblib.load(model_filename)
print(f"Model type: {type(model)}")

Model type: <class 'sklearn.pipeline.Pipeline'>


In [ ]:
prediction = model.predict(sample_house)
prediction_price = float(prediction[0])
print(prediction)

[7968276.12638732]


In [ ]:
prediction_result = {
    "model_file": model_filename,
    "input_file": sample_filename,
    "predicted_price": round(prediction_price, 2)
}

with open("prediction_result.json", "w") as f:
  json.dump(prediction_result, f, indent = 4)

prediction_result

{'model_file': 'house_price_model (1).pkl',
 'input_file': 'sample_house (1).csv',
 'predicted_price': 7968276.13}

In [ ]:
mlflow.set_experiment("housing_price_deployment_validation")

2026/06/27 08:10:13 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/27 08:10:13 INFO mlflow.store.db.utils: Updating database tables
2026/06/27 08:10:16 INFO mlflow.tracking.fluent: Experiment with name 'housing_price_deployment_validation' does not exist. Creating a new experiment.


<Experiment: artifact_location='/content/mlruns/1', creation_time=1782547816905, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1782547816905, lifecycle_stage='active', name='housing_price_deployment_validation', tags={}, trace_location=None, workspace='default'>

In [ ]:
with mlflow.start_run(run_name="model_loaded_and_tested") as run:
  mlflow.log_param("model_file", model_filename)
  mlflow.log_param("sample_file", sample_filename)
  mlflow.log_param("rows_predicted", len(sample_house))

  mlflow.log_metric("predicted_price", prediction_price)
  mlflow.log_artifact("prediction_result.json")
  run_id = run.info.run_id
print(f"Run id {run_id}")

Run id c5415f9f5c094277904203468d13e0e5
